# Unit 05 · Training

The network of unit 04 starts out knowing nothing. Here it learns: a loss that copes with a cube that is 97 % empty,
one training step, the validation Dice, the **overfit-one-batch** test on the laptop, then the first real run on a GPU.

| Part | After video | What you build | Suggested driver |
|---|---|---|---|
| A | 11 (first half) · logits, BCE, soft Dice | the loss | Noa / Adir |
| B | 11 (second half) · the training loop | one training step, the validation Dice, the overfit test | Adir / Noa |
| C | 11 (end) · Colab/Kaggle in practice | the first real run, reading the curves | both |

**Needs:** unit 04 moved into `src/xray3d/nets.py` (`XrayTo3D` and everything it uses), the processed dataset v1
(unit 02) and `splits/split_v1.json` (unit 01). Parts A and B run on the laptop's CPU in a few minutes;
part C's real run needs a GPU (Colab or Kaggle).

In [ ]:
# --- setup (written by Claude, no need to change it): the same cell works locally, on Colab and on Kaggle ---
import os, sys, subprocess
from pathlib import Path

UNIT = "05-training"
REPO_URL = "https://github.com/AdirBuskila/pelvis-xray-3d"
IN_CLOUD = "google.colab" in sys.modules or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_CLOUD:
    root = Path("/content" if "google.colab" in sys.modules else "/kaggle/working") / "pelvis-xray-3d"
    if not root.exists():  # Kaggle: turn Internet on in the notebook settings first
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
else:  # local: the repo is the first folder above this notebook that has a pyproject.toml
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root / "units" / UNIT)
for p in (root / "src", Path.cwd()):  # import xray3d (even before `pip install -e .`) and this unit's checks
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import xray3d
from xray3d import data, env, plotting
import test_unit as T

env.summary()

In [ ]:
import time
import torch
import torch.nn.functional as F
from xray3d import nets, torchdata, trainer

device = "cuda" if torch.cuda.is_available() else "cpu"
sp = torchdata.split_subjects()        # the unit 01 split, restricted to subjects in the processed dataset
print({k: len(v) for k, v in sp.items()}, "subjects in train / val / test;  device:", device)
train_ds = torchdata.XrayDataset(sp["train"], n_views=2, augment=False)
x, y = train_ds[0]
print("one sample:  x", tuple(x.shape), x.dtype, "  y", tuple(y.shape), y.dtype)

---
## Part A · The loss (after video 11, first half)

**What the network sees.** `torchdata` (Claude's code) hands it each image **standardised on its own** (zero mean,
unit std), so the exposure of an X-ray doesn't matter, and unit 07 applies the same function to real X-rays.
Training images also get random **intensity** augmentation. Never geometric: shifting the image without moving the
target cube the same way would teach the network a wrong camera.

In [ ]:
raw = np.load(torchdata.dataset_root() / f"{sp['train'][0]}.npz")["ap"]
rng = np.random.default_rng(0)
augs = [torchdata.standardise(torchdata.augment_intensity(raw, rng)) for _ in range(3)]
plotting.show_row([raw, torchdata.standardise(raw)] + augs,
                  ["raw line integral p", "standardised"] + [f"augmented {i + 1}" for i in range(3)], size=3.2)
s = torchdata.standardise(raw)
print(f"raw: mean {raw.mean():.2f}, std {raw.std():.2f}  ->  standardised: mean {s.mean():.1e}, std {s.std():.3f}")
print("brightness / contrast are not augmented: standardise(a + b * img) == standardise(img) ->",
      np.allclose(torchdata.standardise(3 + 2 * raw), s, atol=1e-4))

**Class imbalance.** How much of the cube is pelvis? And how good does a network look that just says "empty" everywhere?

In [ ]:
occ_all = np.stack([train_ds[i][1][0].numpy() > 0.5 for i in range(len(train_ds))])
frac = occ_all.mean()
print(f"pelvis voxels: {frac:.2%} of the cube (mean over {len(train_ds)} training subjects)")
print(f"predict 'empty' everywhere: voxel accuracy {1 - frac:.2%}, Dice 0.00")
print(f"the prior logit log(f / (1 - f)) = {np.log(frac / (1 - frac)):.2f}  (unit 04 started the head's bias at -4)")

### TODO A1 · `soft_dice_loss`  ·  suggested driver: **Noa**

Dice measures overlap, so it doesn't care how many empty voxels there are. The **soft** version uses the
probabilities p = sigmoid(logits) instead of thresholded 0/1 voxels, so it has a gradient. Compute it **per
sample** and average over the batch; `eps` keeps 0/0 (empty target, empty prediction) from blowing up and makes it a
perfect score instead.

> **Nudge:** flatten each sample to a vector (`.flatten(1)` keeps the batch axis). Dice = (2·Σ p·t + eps) / (Σ p + Σ t + eps) per sample; the loss is 1 − its mean.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def soft_dice_loss(logits, target, eps=1.0):
    """1 - soft Dice, averaged over the batch.

    logits  : (B, 1, D, H, W) raw network output (any real number)
    target  : (B, 1, D, H, W) 0/1 occupancy (float)
    eps     : added to the numerator and the denominator of every sample's Dice
    returns : a 0-dim tensor (so loss.backward() works)
    """
    raise NotImplementedError("TODO A1")

In [ ]:
T.check_soft_dice_loss(soft_dice_loss)

### TODO A2 · `bce_dice_loss`  ·  suggested driver: **Adir**

The proposal's loss: **binary cross-entropy + soft Dice**. BCE scores every voxel on its own (sharp, calibrated
probabilities, but dominated by the 97 % empty ones); Dice scores the overlap of the shape (immune to the imbalance,
but noisy when the prediction is far off). The sum gets both.

> **Nudge:** PyTorch has a BCE that takes logits directly, and it's the numerically safe one: sigmoid first, then log, breaks for confident logits (the check tests a logit of 50).

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def bce_dice_loss(logits, target, w_bce=1.0, w_dice=1.0):
    """w_bce * mean binary cross-entropy (from logits) + w_dice * soft_dice_loss.

    logits, target : (B, 1, D, H, W), target 0/1 float
    returns        : a 0-dim tensor
    """
    raise NotImplementedError("TODO A2")

In [ ]:
T.check_bce_dice_loss(bce_dice_loss)

In [ ]:
# the three parts of the loss for three 'predictions' of one real target
t = y[None]                                              # (1, 1, 64, 64, 64)
guesses = {
    "all empty (logit -10)": torch.full_like(t, -10.0),
    "the prior (logit log f/(1-f))": torch.full_like(t, float(np.log(frac / (1 - frac)))),
    "the truth, blurred": 8 * (F.avg_pool3d(t, 5, stride=1, padding=2) - 0.5),
    "the truth (logits +-10)": 20 * t - 10,
}
print(f"{'prediction':<32}{'BCE':>8}{'soft Dice loss':>16}{'BCE + Dice':>12}")
for name, lg in guesses.items():
    bce = F.binary_cross_entropy_with_logits(lg, t)
    print(f"{name:<32}{float(bce):>8.4f}{float(soft_dice_loss(lg, t)):>16.4f}{float(bce_dice_loss(lg, t)):>12.4f}")

**Explain-back (part A):** "all empty" has a tiny BCE. Why, and why is that a trap? Why does the prior have the
lowest BCE of any *constant* guess? What does the Dice term add, and what would go wrong with Dice alone at the start
of training? Why do we compute Dice per sample and not over the whole batch?

**Part A done.** Move `soft_dice_loss` and `bce_dice_loss` into `src/xray3d/losses.py`.

---
## Part B · One step, and the validation Dice (after video 11, second half)

The training loop itself is Claude's (`xray3d.trainer`: seeds, checkpoints, logging, resume). The two
functions it calls in every iteration are yours.

### TODO B1 · `train_step`  ·  suggested driver: **Adir**

One optimisation step on one batch: forward, loss, backward, update. Return the loss as a plain number for the log.

> **Nudge:** the order is the whole TODO: model in training mode, batch to the device, forget the previous step's gradients, forward, loss, backward, step. Which of these is easy to forget, and what happens if you do?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def train_step(model, batch, optimizer, loss_fn, device="cpu"):
    """One optimisation step.

    model     : the network (put it in training mode: dropout and BatchNorm behave differently in training)
    batch     : (x, y) as a DataLoader yields it, e.g. x (B, V, 128, 128), y (B, 1, 64, 64, 64)
    optimizer : e.g. torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn   : loss_fn(logits, target) -> 0-dim tensor, e.g. bce_dice_loss
    returns   : the loss of this batch (before the update) as a Python float
    """
    raise NotImplementedError("TODO B1")

In [ ]:
T.check_train_step(train_step)

### TODO B2 · `validation_dice`  ·  suggested driver: **Noa**

The number we actually care about: the **Dice of the thresholded prediction** (p > 0.5) on the validation subjects,
averaged over subjects. No gradients, and eval mode (no dropout; a BatchNorm would use its running averages).

> **Nudge:** p > 0.5 is the same as logit > 0 (no sigmoid needed). Per sample: Dice = 2|P ∩ T| / (|P| + |T|), and 1.0 when both are empty. Collect one number per sample across all batches, then average.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def validation_dice(model, loader, device="cpu"):
    """Mean Dice over every sample the loader yields, with occupancy = sigmoid(logits) > 0.5.

    model   : the network (put it in eval mode; no gradients needed)
    loader  : yields (x, y) batches (the validation DataLoader, or any list of (x, y))
    returns : a Python float in [0, 1]; a sample with an empty target and an empty prediction counts as 1.0
    """
    raise NotImplementedError("TODO B2")

In [ ]:
T.check_validation_dice(validation_dice)

### The overfit-one-batch test

Before spending GPU hours, check that the network **can** learn: train on one batch of 2 pelvises, over and
over, nothing else (no augmentation, dropout off: here we *want* it to memorise). A working network memorises the
batch (Dice > 0.9). If it can't, there's a bug (in the loss, the step, the orientation of `lift_view`...), and no
amount of data will fix it. It stops once the Dice passes 0.93, at most 300 steps: about 3–5 minutes on
the laptop.

In [ ]:
torch.manual_seed(0)
xb, yb = torch.utils.data.default_collate([train_ds[i] for i in range(2)])   # one fixed batch
model = nets.XrayTo3D(n_views=2, dropout=0.0).to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
history = {"step": [], "loss": [], "dice": []}
t0 = time.time()
for step in range(1, 300 + 1):
    loss = train_step(model, (xb, yb), opt, bce_dice_loss, device)
    if step % 10 == 0 or step == 1:
        d = validation_dice(model, [(xb, yb)], device)
        history["step"].append(step); history["loss"].append(loss); history["dice"].append(d)
        print(f"step {step:>4}  loss {loss:.4f}  Dice on the batch {d:.3f}  ({time.time() - t0:.0f} s)")
        if d > 0.93:
            break

In [ ]:
T.check_overfit(history)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(history["step"], history["loss"], marker="."); ax[0].set(xlabel="step", ylabel="BCE + soft Dice", title="loss on the one batch", yscale="log")
ax[1].plot(history["step"], history["dice"], marker="."); ax[1].set(xlabel="step", ylabel="Dice", title="Dice on the one batch", ylim=(0, 1))
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

model.eval()
with torch.no_grad():
    pred = (model(xb.to(device))[0, 0] > 0).cpu().numpy()
gt = yb[0, 0].numpy() > 0.5
c = np.argwhere(gt).mean(0).round().astype(int)
show = lambda p: p.T[::-1, ::-1]   # head up; patient's right on the viewer's left
plotting.show_row([show(gt[:, c[1], :]), show(pred[:, c[1], :]), show(gt[c[0], :, :]), show(pred[c[0], :, :])],
                  ["truth, coronal", "memorised, coronal", "truth, sagittal", "memorised, sagittal"], vmin=0, vmax=1, size=3.2)

**Explain-back:** this Dice is on the batch the network trained on. Why does it prove nothing about new patients,
and what does it prove? What would you suspect if the loss went down but the Dice stayed at 0? If the loss went
up?

**Part B done.** Move `train_step` and `validation_dice` into `src/xray3d/training.py`.

---
## Part C · The first real run (after video 11, end)

`trainer.train(name, ...)` runs the loop around your `train_step`: seeded data order and weights, validation Dice every
`eval_every` steps into `log.csv`, `last.pt` every `ckpt_every` steps (and a copy on Google Drive on Colab), `best.pt`
whenever the validation Dice improves, and **resume**: run it again with the same name and it continues from `last.pt`.
The folder is `trainer.run_dir(name)`. The trainer imports your functions from `src/xray3d` (`nets.py`,
`losses.py`, `training.py`), so parts A and B must be moved there first. First a tiny run here on the CPU, just to see the machinery work
(the numbers mean nothing after 6 steps):

In [ ]:
s = trainer.train("smoke-cpu", resume=False, iters=6, batch_size=2, eval_every=3, ckpt_every=3)
print("->", {k: s[k] for k in ("step", "best_val_dice", "final_val_dice")})
s = trainer.train("smoke-cpu", iters=9, batch_size=2, eval_every=3, ckpt_every=3)    # same name: resumes at step 6
print("files:", sorted(p.name for p in trainer.run_dir("smoke-cpu").iterdir()))
print(open(trainer.run_dir("smoke-cpu") / "log.csv").read())

**The real run (GPU only).** Open this notebook on Colab (*Runtime → Change runtime type → T4 GPU*) or on Kaggle
(*Accelerator → GPU T4*, Internet on, the dataset `pelvis-xray3d-v1` attached), and run the cells from the top
(parts A and B take a minute there). On the laptop this cell only prints a message. 4,000 steps of batch 8 take
roughly 10–30 minutes on a T4. Colab: it asks for Drive access, so a disconnected runtime can resume from the copy.

In [ ]:
RUN = "first-2view"
if env.has_gpu():
    if env.where() == "colab":
        from google.colab import drive
        drive.mount("/content/drive")
        from xray3d import dataset
        dataset.ensure_local()                 # unit 02: unzips the processed dataset from Drive
    summary = trainer.train(RUN, n_views=2, seed=0, iters=4000)
    print(summary)
else:
    print("No GPU here: run this cell on Colab or Kaggle. (The laptop would need several hours for 4,000 steps.)")

In [ ]:
runs = [r for r in ("first-2view", "smoke-cpu") if (trainer.run_dir(r) / "log.csv").exists()]
trainer.plot_curves(runs[:1])
plt.show()

**Reading the curves.** Answer these about the GPU run (or, until you've done it, about what you expect):
1. The training loss keeps falling. Does the validation Dice? If it rises and then falls, what is happening, and which
   checkpoint do you keep (`best.pt` or `last.pt`)?
2. The validation set has only a few subjects. How much would you trust a difference of 0.01 in val Dice?
3. The run used seed 0. What would change with seed 1, and why does unit 06 run three seeds per arm?
4. Where does this Dice sit compared with unit 03's baselines (visual hull, nearest neighbour)?

**From the command line** (what unit 06 runs on Kaggle):
`python units/05-training/train.py --views 2 --seed 0 --iters 4000 --name v2-s0`

**Part C done.** Write the run's best val Dice in `LOG.md`. Then run pytest:

In [ ]:
!{sys.executable} -m pytest -q -rs .

Fill in `LOG.md` (driver per TODO, hints, explain-back answers, the overfit numbers, the first run's best val Dice).
Next: **unit 06**, the controlled experiment.